# Validação das regras de qualidade RAW → CORE

Evidência reproduzível das regras definidas para a M05-03. O script não
altera os CSVs nem o banco de dados.

In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path

import pandas as pd


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError(f"Raiz do repositório não encontrada a partir de {start}")


ROOT = find_repository_root(Path.cwd())
DATA_DIR = ROOT / "data/raw"
OUTPUT_DIR = ROOT / "outputs/data-loading"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

UF = {
    "AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO", "MA", "MG",
    "MS", "MT", "PA", "PB", "PE", "PI", "PR", "RJ", "RN", "RO", "RR",
    "RS", "SC", "SE", "SP", "TO",
}
ORDER_STATUS = {
    "approved", "canceled", "created", "delivered", "invoiced",
    "processing", "shipped", "unavailable",
}
PAYMENT_TYPE = {"boleto", "credit_card", "debit_card", "not_defined", "voucher"}
ID_PATTERN = r"[0-9a-f]{32}"
CEP_PATTERN = r"[0-9]{5}"
INTEGER_PATTERN = r"[0-9]+"
DECIMAL_PATTERN = r"[0-9]+(?:\.[0-9]{1,2})?"


def load(filename: str, usecols: list[str] | None = None) -> pd.DataFrame:
    return pd.read_csv(
        DATA_DIR / filename,
        dtype=str,
        encoding="utf-8-sig",
        keep_default_na=False,
        na_values=[""],
        usecols=usecols,
    )


results: list[dict[str, object]] = []


def record(
    rule_id: str,
    scope: str,
    description: str,
    violations: int,
    classification: str,
    treatment: str,
) -> None:
    results.append(
        {
            "rule_id": rule_id,
            "scope": scope,
            "description": description,
            "violations": int(violations),
            "classification": classification,
            "treatment": treatment,
        }
    )


def required(rule_id: str, scope: str, series: pd.Series) -> None:
    record(rule_id, scope, "Valor obrigatório", series.isna().sum(), "bloqueante", "bloquear carga e registrar exceção")


def regex(rule_id: str, scope: str, series: pd.Series, pattern: str) -> None:
    invalid = series.notna() & ~series.str.fullmatch(pattern)
    record(rule_id, scope, f"Formato {pattern}", invalid.sum(), "bloqueante", "bloquear carga e registrar exceção")


def domain(rule_id: str, scope: str, series: pd.Series, values: set[str]) -> None:
    invalid = series.notna() & ~series.isin(values)
    record(rule_id, scope, "Domínio aprovado", invalid.sum(), "bloqueante", "bloquear carga e registrar exceção")


def max_length(rule_id: str, scope: str, series: pd.Series, limit: int) -> None:
    invalid = series.notna() & (series.str.len() > limit)
    record(rule_id, scope, f"Comprimento máximo {limit}", invalid.sum(), "bloqueante", "bloquear carga e registrar exceção")


def nonempty(rule_id: str, scope: str, series: pd.Series) -> None:
    invalid = series.notna() & series.str.strip().eq("")
    record(rule_id, scope, "Texto não vazio", invalid.sum(), "bloqueante", "bloquear carga e registrar exceção")


def unique(rule_id: str, scope: str, frame: pd.DataFrame, columns: list[str]) -> None:
    violations = frame.duplicated(subset=columns, keep=False).sum()
    record(rule_id, scope, "Unicidade da chave", violations, "bloqueante", "bloquear carga e registrar exceção")


def timestamp(rule_id: str, scope: str, series: pd.Series) -> None:
    converted = pd.to_datetime(series, errors="coerce", format="%Y-%m-%d %H:%M:%S")
    invalid = series.notna() & converted.isna()
    record(rule_id, scope, "Timestamp no formato da fonte", invalid.sum(), "bloqueante", "bloquear carga e registrar exceção")


def numeric_range(
    rule_id: str,
    scope: str,
    series: pd.Series,
    minimum: float,
    maximum: float | None = None,
    pattern: str | None = None,
) -> None:
    converted = pd.to_numeric(series, errors="coerce")
    invalid = series.notna() & converted.isna()
    if pattern:
        invalid |= series.notna() & ~series.str.fullmatch(pattern)
    invalid |= converted.notna() & (converted < minimum)
    if maximum is not None:
        invalid |= converted.notna() & (converted > maximum)
    record(rule_id, scope, "Conversão numérica e intervalo", invalid.sum(), "bloqueante", "bloquear carga e registrar exceção")


def foreign_key(
    rule_id: str,
    scope: str,
    child: pd.Series,
    parent_values: set[str],
) -> None:
    invalid = child.notna() & ~child.isin(parent_values)
    record(rule_id, scope, "Integridade referencial", invalid.sum(), "bloqueante", "bloquear carga e registrar exceção")

In [2]:
customers = load("olist_customers_dataset.csv")
orders = load("olist_orders_dataset.csv")
products = load("olist_products_dataset.csv")
sellers = load("olist_sellers_dataset.csv")
items = load("olist_order_items_dataset.csv")
payments = load("olist_order_payments_dataset.csv")
reviews = load("olist_order_reviews_dataset.csv")
translations = load("product_category_name_translation.csv")

frames = {
    "customers": customers,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "items": items,
    "payments": payments,
    "reviews": reviews,
}

required_columns = {
    "customers": list(customers.columns),
    "orders": [
        "order_id", "customer_id", "order_status", "order_purchase_timestamp",
        "order_estimated_delivery_date",
    ],
    "products": ["product_id"],
    "sellers": list(sellers.columns),
    "items": list(items.columns),
    "payments": list(payments.columns),
    "reviews": [
        "review_id", "order_id", "review_score", "review_creation_date",
        "review_answer_timestamp",
    ],
}
for table, columns in required_columns.items():
    for column in columns:
        required(f"NUL-{table}-{column}", f"{table}.{column}", frames[table][column])

id_columns = {
    "customers": ["customer_id", "customer_unique_id"],
    "orders": ["order_id", "customer_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "items": ["order_id", "product_id", "seller_id"],
    "payments": ["order_id"],
    "reviews": ["review_id", "order_id"],
}
for table, columns in id_columns.items():
    for column in columns:
        regex(f"FMT-{table}-{column}", f"{table}.{column}", frames[table][column], ID_PATTERN)

for table, column in [
    ("customers", "customer_zip_code_prefix"),
    ("sellers", "seller_zip_code_prefix"),
]:
    regex(f"FMT-{table}-cep", f"{table}.{column}", frames[table][column], CEP_PATTERN)

for table, column in [
    ("customers", "customer_city"),
    ("sellers", "seller_city"),
]:
    nonempty(f"TXT-{table}-cidade", f"{table}.{column}", frames[table][column])
    max_length(f"LEN-{table}-cidade", f"{table}.{column}", frames[table][column], 50)

domain("DOM-customers-uf", "customers.customer_state", customers["customer_state"], UF)
domain("DOM-sellers-uf", "sellers.seller_state", sellers["seller_state"], UF)
domain("DOM-orders-status", "orders.order_status", orders["order_status"], ORDER_STATUS)
domain("DOM-payments-type", "payments.payment_type", payments["payment_type"], PAYMENT_TYPE)

unique("KEY-customers", "customers.customer_id", customers, ["customer_id"])
unique("KEY-orders-id", "orders.order_id", orders, ["order_id"])
unique("KEY-orders-customer", "orders.customer_id", orders, ["customer_id"])
unique("KEY-products", "products.product_id", products, ["product_id"])
unique("KEY-sellers", "sellers.seller_id", sellers, ["seller_id"])
unique("KEY-items", "items.(order_id,order_item_id)", items, ["order_id", "order_item_id"])
unique("KEY-payments", "payments.(order_id,payment_sequential)", payments, ["order_id", "payment_sequential"])
unique("KEY-reviews", "reviews.(review_id,order_id)", reviews, ["review_id", "order_id"])

for column in [
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
]:
    timestamp(f"TMP-orders-{column}", f"orders.{column}", orders[column])
timestamp("TMP-items-shipping", "items.shipping_limit_date", items["shipping_limit_date"])
timestamp("TMP-reviews-creation", "reviews.review_creation_date", reviews["review_creation_date"])
timestamp("TMP-reviews-answer", "reviews.review_answer_timestamp", reviews["review_answer_timestamp"])

numeric_range("NUM-items-id", "items.order_item_id", items["order_item_id"], 1, 32767, INTEGER_PATTERN)
numeric_range("NUM-items-price", "items.price", items["price"], 0, 9999999999.99, DECIMAL_PATTERN)
numeric_range("NUM-items-freight", "items.freight_value", items["freight_value"], 0, 9999999999.99, DECIMAL_PATTERN)
numeric_range("NUM-payments-sequential", "payments.payment_sequential", payments["payment_sequential"], 1, 32767, INTEGER_PATTERN)
numeric_range("NUM-payments-installments", "payments.payment_installments", payments["payment_installments"], 0, 32767, INTEGER_PATTERN)
numeric_range("NUM-payments-value", "payments.payment_value", payments["payment_value"], 0, 9999999999.99, DECIMAL_PATTERN)
numeric_range("NUM-reviews-score", "reviews.review_score", reviews["review_score"], 1, 5, INTEGER_PATTERN)

for column in ["product_name_lenght", "product_description_lenght", "product_photos_qty"]:
    numeric_range(f"NUM-products-{column}", f"products.{column}", products[column], 0, 32767, INTEGER_PATTERN)
numeric_range("NUM-products-weight", "products.product_weight_g", products["product_weight_g"], 0, 2147483647, INTEGER_PATTERN)
for column in ["product_length_cm", "product_height_cm", "product_width_cm"]:
    numeric_range(f"NUM-products-{column}", f"products.{column}", products[column], 0, 32767, INTEGER_PATTERN)
max_length("LEN-products-category", "products.product_category_name", products["product_category_name"], 50)
nonempty("TXT-products-category", "products.product_category_name", products["product_category_name"])

foreign_key("REF-orders-customer", "orders.customer_id→customers.customer_id", orders["customer_id"], set(customers["customer_id"]))
foreign_key("REF-items-order", "items.order_id→orders.order_id", items["order_id"], set(orders["order_id"]))
foreign_key("REF-items-product", "items.product_id→products.product_id", items["product_id"], set(products["product_id"]))
foreign_key("REF-items-seller", "items.seller_id→sellers.seller_id", items["seller_id"], set(sellers["seller_id"]))
foreign_key("REF-payments-order", "payments.order_id→orders.order_id", payments["order_id"], set(orders["order_id"]))
foreign_key("REF-reviews-order", "reviews.order_id→orders.order_id", reviews["order_id"], set(orders["order_id"]))

In [3]:
geolocation = load("olist_geolocation_dataset.csv")
for column in geolocation.columns:
    required(f"NUL-geolocation-{column}", f"geolocation.{column}", geolocation[column])
regex("FMT-geolocation-cep", "geolocation.geolocation_zip_code_prefix", geolocation["geolocation_zip_code_prefix"], CEP_PATTERN)
numeric_range("NUM-geolocation-lat", "geolocation.geolocation_lat", geolocation["geolocation_lat"], -90, 90)
numeric_range("NUM-geolocation-lng", "geolocation.geolocation_lng", geolocation["geolocation_lng"], -180, 180)
nonempty("TXT-geolocation-city", "geolocation.geolocation_city", geolocation["geolocation_city"])
max_length("LEN-geolocation-city", "geolocation.geolocation_city", geolocation["geolocation_city"], 50)
domain("DOM-geolocation-uf", "geolocation.geolocation_state", geolocation["geolocation_state"], UF)

duplicate_geolocation = int(geolocation.duplicated().sum())
record(
    "DUP-geolocation-exact",
    "geolocation.*",
    "Duplicidade integral além da primeira ocorrência",
    duplicate_geolocation,
    "transformacao",
    "deduplicar por todas as colunas e reconciliar",
)

cep_values = set(customers["customer_zip_code_prefix"].dropna())
cep_values.update(sellers["seller_zip_code_prefix"].dropna())
cep_values.update(geolocation["geolocation_zip_code_prefix"].dropna())
record("DRV-prefixo-cep", "core.prefixo_cep", "União distinta de prefixos válidos", 0, "transformacao", f"gerar {len(cep_values)} prefixos")

record("ACC-products-nullable", "products.*", "Ausências permitidas no CORE", int(products.isna().sum().sum()), "informativa", "preservar NULL")
record("ACC-orders-nullable", "orders.*", "Ausências temporais permitidas", int(orders.isna().sum().sum()), "informativa", "preservar NULL")
record("ACC-payments-zero-installments", "payments.payment_installments", "Parcelas iguais a zero permitidas", int(payments["payment_installments"].eq("0").sum()), "informativa", "preservar valor")
record("ACC-reviews-comments", "reviews.review_comment_*", "Comentários fora do CORE", int(reviews[["review_comment_title", "review_comment_message"]].isna().sum().sum()), "informativa", "manter somente na RAW")
record("ACC-customers-unique-repeat", "customers.customer_unique_id", "Repetição permitida fora da PK", int(customers.duplicated("customer_unique_id").sum()), "informativa", "preservar ocorrências")
record("ACC-reviews-id-repeat", "reviews.review_id", "Repetição permitida pela PK composta", int(reviews.duplicated("review_id").sum()), "informativa", "preservar ocorrências")
record("ACC-payments-not-defined", "payments.payment_type", "Tipo not_defined aceito no domínio", int(payments["payment_type"].eq("not_defined").sum()), "informativa", "preservar valor")
missing_translations = set(products["product_category_name"].dropna()) - set(translations["product_category_name"].dropna())
record("ACC-category-translation", "product_category_name_translation", "Categorias de produto sem tradução", int(products["product_category_name"].isin(missing_translations).sum()), "informativa", "manter categoria original no CORE")

In [4]:
blocking_violations = sum(
    int(result["violations"])
    for result in results
    if result["classification"] == "bloqueante"
)
summary = {
    "rules_evaluated": len(results),
    "blocking_violations": blocking_violations,
    "geolocation_duplicates_beyond_first": duplicate_geolocation,
    "distinct_postal_prefixes": len(cep_values),
    "expected_core_rows": {
        "prefixo_cep": len(cep_values),
        "cliente": len(customers),
        "produto": len(products),
        "vendedor": len(sellers),
        "pedido": len(orders),
        "item_pedido": len(items),
        "pagamento": len(payments),
        "avaliacao": len(reviews),
        "geolocalizacao": len(geolocation) - duplicate_geolocation,
    },
    "result": "APROVADO" if blocking_violations == 0 else "REPROVADO",
    "rules": results,
}
OUTPUT_DIR.joinpath("raw_to_core_quality_report.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)

print(f"Regras avaliadas: {summary['rules_evaluated']}")
print(f"Violações bloqueantes: {blocking_violations}")
print(f"Duplicatas integrais de geolocalização: {duplicate_geolocation:,}".replace(",", "."))
print(f"Prefixos de CEP distintos derivados: {len(cep_values):,}".replace(",", "."))
print(f"VALIDAÇÃO RAW → CORE: {summary['result']}")

if blocking_violations:
    raise SystemExit("Existem violações bloqueantes no snapshot analisado")

Regras avaliadas: 113
Violações bloqueantes: 0
Duplicatas integrais de geolocalização: 261.831
Prefixos de CEP distintos derivados: 19.177
VALIDAÇÃO RAW → CORE: APROVADO
